# Stacked gravitational potential around halos: Gadget-4 vs pkdgrav3

Mean peculiar potential $\langle\phi\rangle(r)$ (physical, (km/s)$^2$) in shells of comoving distance $r$ (Mpc/h) from
host-halo centres, stacked in bins of $M_{\rm vir}$, for the Gadget-4 and pkdgrav3 runs started from the same ICs
(Sahyadri default settings, pawna). Made by `scripts/gadget_pkdgrav3_compare/halo_potential_profile.py`: only the stacks
are stored (no per-halo profiles).

* shaded band: $\pm$ the standard deviation of $\phi$ over all (halo, particle) pairs in the shell (`stack_std`)
* error bars: jackknife error of the mean over $3^3$ sub-cubes of halo centres (`stack_err`)
* each code uses its own consistent-trees halos (Rockstar `.list` where there is none: 32$^3$); pkdgrav3 without its
  softened self-potential and aligned to Gadget's zero point by the particle median offset (`--offset`)
* `pkc`: the same pkdgrav3 run from the compressed `.pkc` snapshot (potential quantised to the `.pkc` step)

Set `RUNS` to the comparison directories (copied from pawna, see the report) and run all cells.

In [ ]:
import glob, os, re
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

HERE = os.path.abspath('.')
RUNS = {'256^3': os.path.join(HERE, 'data', 'pkd256_vs_gadget256_r1', 'profiles'),
        '32^3': os.path.join(HERE, 'data', 'pkd32_vs_gadget32_r1', 'profiles')}
FIGS = os.path.join(HERE, 'figs')
os.makedirs(FIGS, exist_ok=True)
STYLE = {'gadget': dict(color='#1f5fa8', label='Gadget-4'),
         'pkd': dict(color='#c8553d', label='pkdgrav3 (HDF5)'),
         'pkc': dict(color='#333333', label='pkdgrav3 (.pkc)', ls='--')}

def load(d):
    out = {}
    for f in sorted(glob.glob(os.path.join(d, '*_out*.npz'))):
        kind, i = re.match(r'(\w+)_out(\d+)\.npz', os.path.basename(f)).groups()
        out.setdefault(int(i), {})[kind] = dict(np.load(f, allow_pickle=True))
    return out

P = {k: load(v) for k, v in RUNS.items() if os.path.isdir(v)}
for k, v in P.items():
    print(k, 'outputs', sorted(v), '; z =', [round(float(v[i]['gadget']['z']), 3) for i in sorted(v)])

## Mean potential and its scatter, per mass bin, for every output

In [ ]:
def mlabel(lo, hi):
    f = lambda m: '10^{%.1f}' % np.log10(m)
    return '$%s \\leq M_{\\rm vir} < %s$' % (f(lo), f(hi))

def plot_output(run, i, save=True):
    S = P[run][i]
    g = S['gadget']
    me, nb = g['m_edges'], len(g['n_halos'])
    fig, ax = plt.subplots(2, nb, figsize=(4.2 * nb, 6.4), sharex=True, squeeze=False,
                           gridspec_kw=dict(height_ratios=[2.2, 1]))
    for b in range(nb):
        a0, a1 = ax[0, b], ax[1, b]
        for kind in ('gadget', 'pkd', 'pkc'):
            if kind not in S or S[kind]['n_halos'][b] == 0:
                continue
            s, st = S[kind], STYLE[kind]
            r = s['r_mean'][b]
            m, sd, e = s['stack_mean'][b] / 1e5, s['stack_std'][b] / 1e5, s['stack_err'][b] / 1e5
            a0.errorbar(r, m, e, color=st['color'], ls=st.get('ls', '-'), lw=1.3, capsize=0,
                        label='%s (%d)' % (st['label'], s['n_halos'][b]))
            if kind != 'pkc':
                a0.fill_between(r, m - sd, m + sd, color=st['color'], alpha=0.15, lw=0)
            if kind != 'gadget':
                rg, mg = g['r_mean'][b], g['stack_mean'][b]
                dm = (s['stack_mean'][b] - mg) / 1e5
                a1.plot(r, dm, color=st['color'], ls=st.get('ls', '-'))
                a1.fill_between(r, dm - np.hypot(s['stack_err'][b], g['stack_err'][b]) / 1e5,
                                dm + np.hypot(s['stack_err'][b], g['stack_err'][b]) / 1e5, color=st['color'], alpha=0.2, lw=0)
        a0.axhline(0, color='k', lw=0.5); a1.axhline(0, color='k', lw=0.5)
        a0.set_title(mlabel(me[b], me[b + 1]), fontsize=10)
        a1.set_xscale('log'); a1.set_xlabel('r  [comoving Mpc/h]')
        if a0.get_legend_handles_labels()[0]:
            a0.legend(fontsize=7, loc='lower right')
    ax[0, 0].set_ylabel(r'$\langle\phi\rangle \pm \sigma_\phi$  [$10^5$ (km/s)$^2$]')
    ax[1, 0].set_ylabel(r'pkdgrav3 $-$ Gadget-4  [$10^5$ (km/s)$^2$]')
    fig.suptitle('%s, z = %.2f: stacked potential around host halos' % (run, float(g['z'])))
    fig.tight_layout()
    if save:
        fig.savefig(os.path.join(FIGS, 'stack_%s_out%03d.pdf' % (run.replace('^3', ''), i)))
    return fig

for run in P:
    for i in sorted(P[run]):
        if any(P[run][i]['gadget']['n_halos']):
            plot_output(run, i)
plt.show()

## Difference of the mean potential in units of the shell scatter, all outputs

$\Delta\langle\phi\rangle/\sigma_\phi$ = (pkdgrav3 $-$ Gadget-4) / Gadget-4 `stack_std`; the band is the jackknife error. (A ratio of the means is ill-defined where $\langle\phi\rangle$ crosses zero at large $r$.)

In [ ]:
def plot_relative(run, save=True):
    outs = [i for i in sorted(P[run]) if any(P[run][i]['gadget']['n_halos'])]
    g0 = P[run][outs[-1]]['gadget']; nb = len(g0['n_halos'])
    fig, ax = plt.subplots(1, nb, figsize=(4.2 * nb, 3.6), sharey=True, squeeze=False)
    cols = plt.cm.viridis(np.linspace(0, 0.85, len(outs)))
    for c, i in zip(cols, outs):
        g, p = P[run][i]['gadget'], P[run][i]['pkd']
        for b in range(nb):
            if g['n_halos'][b] == 0 or p['n_halos'][b] == 0:
                continue
            rel = (p['stack_mean'][b] - g['stack_mean'][b]) / g['stack_std'][b]
            err = np.hypot(p['stack_err'][b], g['stack_err'][b]) / g['stack_std'][b]
            ax[0, b].plot(g['r_mid'], rel, color=c, label='z = %.2f' % float(g['z']))
            ax[0, b].fill_between(g['r_mid'], rel - err, rel + err, color=c, alpha=0.2, lw=0)
    for b in range(nb):
        ax[0, b].axhline(0, color='k', lw=0.5); ax[0, b].set_xscale('log')
        ax[0, b].set_title(mlabel(g0['m_edges'][b], g0['m_edges'][b + 1]), fontsize=10)
        ax[0, b].set_xlabel('r  [comoving Mpc/h]')
    ax[0, 0].set_ylabel(r'$(\langle\phi\rangle_{\rm pkd} - \langle\phi\rangle_{\rm Gadget}) / \sigma_\phi$')
    ax[0, 0].set_ylim(-1, 1)
    for a in ax[0]:
        if a.get_legend_handles_labels()[0]:
            a.legend(fontsize=8); break
    fig.suptitle(run + ': difference of the stacked mean potential / shell scatter')
    fig.tight_layout()
    if save:
        fig.savefig(os.path.join(FIGS, 'stack_rel_%s.pdf' % run.replace('^3', '')))
    return fig

for run in P:
    plot_relative(run)
plt.show()

## Matter overdensity around the same halos (halo-matter cross-correlation)

The pair counts of the same stacks give $1+\xi_{hm}(r)$; differences here show where the two codes put the mass.

In [ ]:
def plot_delta(run, save=True):
    outs = [i for i in sorted(P[run]) if any(P[run][i]['gadget']['n_halos'])]
    i = outs[-1]; S = P[run][i]; g = S['gadget']; nb = len(g['n_halos'])
    fig, ax = plt.subplots(1, nb, figsize=(4.2 * nb, 3.6), squeeze=False)
    for b in range(nb):
        for kind in ('gadget', 'pkd'):
            s = S[kind]
            if s['n_halos'][b] == 0:
                continue
            ax[0, b].errorbar(s['r_mean'][b], 1 + s['delta_hm'][b], s['delta_hm_err'][b], color=STYLE[kind]['color'],
                              label=STYLE[kind]['label'], capsize=0)
        ax[0, b].set_xscale('log'); ax[0, b].set_yscale('log'); ax[0, b].set_xlabel('r  [comoving Mpc/h]')
        ax[0, b].set_title(mlabel(g['m_edges'][b], g['m_edges'][b + 1]), fontsize=10)
    ax[0, 0].set_ylabel(r'$1 + \xi_{hm}(r)$'); ax[0, 0].legend(fontsize=8)
    fig.suptitle('%s, z = %.2f' % (run, float(g['z']))); fig.tight_layout()
    if save:
        fig.savefig(os.path.join(FIGS, 'xihm_%s.pdf' % run.replace('^3', '')))
    return fig

for run in P:
    plot_delta(run)
plt.show()

## Table: mean potential difference at a few radii (pkdgrav3 HDF5 vs Gadget-4), relative and in units of $\sigma_\phi$

In [ ]:
for run in P:
    for i in sorted(P[run]):
        g, p = P[run][i]['gadget'], P[run][i]['pkd']
        for b in range(len(g['n_halos'])):
            if g['n_halos'][b] == 0 or p['n_halos'][b] == 0:
                continue
            rel = p['stack_mean'][b] / g['stack_mean'][b] - 1
            ds = (p['stack_mean'][b] - g['stack_mean'][b]) / g['stack_std'][b]
            pick = [np.nanargmin(abs(g['r_mid'] - r0)) for r0 in (0.05, 0.2, 1, 5, 20, 45)]
            print('%-6s z=%.2f  %s  N=%5d/%5d  ' % (run, float(g['z']), mlabel(g['m_edges'][b], g['m_edges'][b+1]),
                  g['n_halos'][b], p['n_halos'][b]) + '  '.join('r=%.2g: %+.3f (%+.2f sig)' % (g['r_mid'][k], rel[k], ds[k]) for k in pick))